# Limpando o arquivo .csv

In [ ]:
DetectorFactory.seed = 0  # langdetect não é determinístico sem seed fixa

class MojibakeDetector:
    """Detector robusto de texto corrompido (mojibake)
    """

    PATTERN_SMART_QUOTES_BROKEN = re.compile(r'â€')
    PATTERN_LATIN1_SCRIPT_MIXED = re.compile(r'[\xC0-\xFF]{2,}')
    PATTERN_UTF8_CONTROL_BYTES = re.compile(r'Ã[\x80-\xbf]|Â[\x80-\xbf]|Ð[\x80-\xbf]|Ñ[\x80-\xbf]')

    @classmethod
    def has_mojibake(cls, text):
        if not isinstance(text, str) or len(text) == 0:
            return False

        if cls.PATTERN_SMART_QUOTES_BROKEN.search(text):
            return True

        if cls.PATTERN_LATIN1_SCRIPT_MIXED.search(text):
            return True

        if cls.PATTERN_UTF8_CONTROL_BYTES.search(text):
            return True

        return False


def detectar_idioma(text):
    """Detecta o idioma de um texto usando langdetect
    """
    if not isinstance(text, str) or len(text) < 10:
        return None

    try:
        lang = detect(text)
        return lang
    except:
        return None


def transcricao_video(text):
    """
    Detecta se o texto é uma transcrição de vídeo (YouTube, etc.)

    Sinais de vídeo:
    - Padrão de timestamp (HH:MM:SS ou MM:SS)
    - Densidade alta de timestamps
    - Começa com timestamp
    """
    if not isinstance(text, str) or len(text) < 50:
        return False

    # Padrão de timestamp: HH:MM:SS ou MM:SS ou H:MM:SS
    timestamp_pattern = re.compile(r'\d{1,2}:\d{2}:\d{2}|\d{1,3}:\d{2}')

    timestamps = timestamp_pattern.findall(text)

    if len(timestamps) == 0:
        return False

    # Densidade de timestamps (quantos timestamps por 1000 caracteres)
    densidade = len(timestamps) / (len(text) / 1000)

    # Se há mais de 2 timestamps por 1000 caracteres, é vídeo
    if densidade > 2:
        return True

    # Se começa com timestamp, é bem provável
    if text.lstrip().startswith(('0:0', '0:', '1:', '2:', '3:', '4:', '5:', '6:', '7:', '8:', '9:')):
        if len(timestamps) >= 5:
            return True

    return False


def remover_ruido_html(text):
    """Remove tags HTML/JATS residuais
    """
    if pd.isna(text):
        return text
    text = str(text)
    text = html.unescape(text)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


print("INICIANDO PIPELINE DE LIMPEZA COM DETECÇÃO DE IDIOMA...")
print("="*80)

print("\n[1/7] Carregando dataset original...")
df_init = pd.read_csv(f"{NAME_FILE}.csv", encoding='utf-8')
print(f"     Total de artigos: {len(df_init)}")

print("\n[2/7] Descartando resumos vazios (NaN)...")
df_clean = df_init.dropna(subset=['abstract']).copy()
discard_nan = len(df_init) - len(df_clean)
print(f"     Artigos descartados: {discard_nan}")
print(f"     Artigos com abstract válido: {len(df_clean)}")

print("\n[3/7] Removendo marcação HTML/JATS residual...")
count_html = df_clean['abstract'].str.contains(r'<[^>]+>', regex=True, na=False).sum()
df_clean['abstract'] = df_clean['abstract'].apply(remover_ruido_html)
print(f"     Artigos que continham tags HTML: {count_html}")

print("\n[4/7] Detectando e removendo artigos com mojibake...")
mask_mojibake = df_clean['abstract'].apply(MojibakeDetector.has_mojibake)
count_mojibake = mask_mojibake.sum()
print(f"     Artigos com encoding corrompido detectado: {count_mojibake} ({count_mojibake / len(df_clean) * 100:.2f}%)")
df_clean = df_clean[~mask_mojibake].copy()
print(f"     Artigos após remoção de mojibake: {len(df_clean)}")

print("\n[5/7] Detectando e removendo artigos NÃO-INGLESES...")
print("     (Isso pode levar um tempo...)")
language_detection = df_clean['abstract'].apply(detectar_idioma)
mask_not_english = language_detection != 'en'
count_not_english = mask_not_english.sum()

# Conta a distribuição de idiomas detectados
languages_detected = language_detection[language_detection.notna()].value_counts()
print(f"\n     Distribuição de idiomas detectados:")
for language, count in languages_detected.head(10).items():
    print(f"       {language}: {count}")

print(f"\n     Artigos NÃO-INGLESES: {count_not_english} ({count_not_english / len(df_clean) * 100:.2f}%)")
df_clean = df_clean[~mask_not_english].copy()
print(f"     Artigos após remoção de não-ingleses: {len(df_clean)}")

print("\n[6/7] Detectando e removendo transcrições de vídeo...")
mask_video = df_clean['abstract'].apply(transcricao_video)
count_video = mask_video.sum()
print(f"     Transcrições de vídeo detectadas: {count_video} ({count_video / len(df_clean) * 100:.2f}%)")
df_clean = df_clean[~mask_video].copy()
print(f"     Artigos após remoção de transcrições: {len(df_clean)}")

print("\n[7/7] Finalizando dataset...")
df_clean = df_clean.reset_index(drop=True)

print("\n" + "="*80)
print("RELATÓRIO FINAL")
print("="*80)
print(f"Total de artigos antes da limpeza: {len(df_init)}")
print(f"Artigos descartados:")
print(f"  - Resumos vazios: {discard_nan}")
print(f"  - Mojibake: {count_mojibake}")
print(f"  - Não-inglês: {count_not_english}")
print(f"  - Transcrição de vídeo: {count_video}")
print(f"Total de modificações em artigos:")
print(f"  - Tags HTML removidas: {count_html}")
print(f"\nArtigos finais: {len(df_clean)}")
print(f"Taxa de retenção: {len(df_clean) / len(df_init) * 100:.2f}%")

NAME_FILE_2 = "insira-o-nome-do-segundo-arquivo-csv"

df_clean.to_csv(f"{NAME_FILE_2}.csv", index=False, encoding='utf-8')
print(f"\nDataset salvo: {NAME_FILE_2}.csv")
